# WildChat In-Depth Insights Analysis

Exploratory analysis of topics, commercial content (brands, intent), themes, and topic modeling for the [WildChat](https://huggingface.co/datasets/allenai/WildChat) dataset. If you have run the **Export English-only** section in `explore.ipynb`, data is loaded from `data/english_chunks/` (English-only, chunked parquet); otherwise it loads from Hugging Face and samples. Uses `insights_utils` for logic; optional OpenAI API for purpose/theme/brand labeling (set `USE_OPENAI = True` and add `OPENAI_API_KEY` to `.env`). Outputs are saved under `insights_output/` for reruns.

In [1]:
# Configuration (edit and run first)
SAMPLE_N = 50000           # number of conversations; set to None to use SAMPLE_PCT
SAMPLE_PCT = None         # e.g. 0.01; used only when SAMPLE_N is None
LOAD_FROM_SAVED = False   # if True, load saved tables from INSIGHTS_OUTPUT_DIR where possible
INSIGHTS_OUTPUT_DIR = "insights_output"
ENGLISH_CHUNKS_DIR = "data/english_chunks"  # if this folder has parquet chunks, load from here (run explore.ipynb export first)
USE_OPENAI = True        # set True to run purpose/theme/brand labeling (requires OPENAI_API_KEY in .env)
RANDOM_SEED = 42
N_TOPICS = 15             # for topic modeling (NMF/LDA)

In [2]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

from eda_utils import sample_by_conversation, load_english_chunked_parquet
from insights_utils import (
    extract_text_column,
    add_commercial_intent_heuristic,
    ensure_output_dir,
    load_or_build,
    run_nmf,
    run_lda,
    get_top_terms_nmf,
    get_top_terms_lda,
    assign_dominant_topic,
    build_topic_distribution_df,
    theme_counts_over_time,
    underserved_metrics,
    label_purpose_openai,
    extract_brands_openai,
    label_theme_openai,
    label_theme_openai_parallel,
    label_commercial_intent_openai,
)
load_dotenv()

/Users/Larry.Jin/miniconda3/envs/wildchat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# Check if OpenAI API key is set and works (optional; run before using USE_OPENAI=True)
import os
key = os.environ.get("OPENAI_API_KEY")
if not key or not key.strip():
    print("OPENAI_API_KEY is not set (add it to .env or environment). USE_OPENAI will not work.")
else:
    try:
        from openai import OpenAI
        client = OpenAI()
        r = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "Say OK"}], max_tokens=5)
        reply = (r.choices[0].message.content or "").strip()
        print("OpenAI API key works. Reply:", reply)
    except Exception as e:
        print("OpenAI API key check failed:", e)

OpenAI API key works. Reply: OK!


In [4]:
# Load from English chunked parquet (if available) else from Hugging Face; then sample
chunks_dir = Path(ENGLISH_CHUNKS_DIR)
if chunks_dir.exists() and list(chunks_dir.glob("english_*.parquet")):
    df_full = load_english_chunked_parquet(chunks_dir)
    n_total = len(df_full)
    n = SAMPLE_N if SAMPLE_N is not None else max(1, int(n_total * (SAMPLE_PCT or 0.01)))
    n = min(n, n_total)
    df = df_full.sample(n=n, random_state=RANDOM_SEED)
    print(f"Loaded {n_total} English conversations from {chunks_dir}; sampled {len(df)}.")
else:
    dataset = load_dataset("allenai/WildChat", split="train")
    sampled = sample_by_conversation(dataset, n=SAMPLE_N, pct=SAMPLE_PCT, seed=RANDOM_SEED)
    df = sampled.to_pandas()
    print(f"Sampled {len(df)} conversations from Hugging Face (no English chunks at {chunks_dir}).")
df["first_user_text"] = extract_text_column(df, mode="first_user")
print(f"Columns: {list(df.columns)}")
df.head(2)

Loaded 284168 English conversations from data/english_chunks; sampled 50000.
Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'first_user_text']


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,first_user_text
229795,10517d64ed2898af190f051690a753ca,gpt-3.5-turbo,2023-09-20 13:42:50+00:00,[{'content': 'Write a product description abou...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00012704751861747354, '...",False,False,Write a product description about a brand Fhey...
124870,69a7e487aa5b1fb131a78735c8482fd7,gpt-3.5-turbo,2023-06-27 18:39:45+00:00,[{'content': 'Write dialogue from a scene from...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00012649749987758696, '...",False,False,Write dialogue from a scene from the animated ...


In [5]:
import textwrap

def print_conversation(conv, width=100, title=None):
    """Print one conversation with wrapped lines so full content is visible on screen."""
    if title:
        print(title)
        print("-" * min(len(title), width))
    # Accept list or numpy array (e.g. from df["conversation"])
    if not isinstance(conv, list):
        try:
            conv = list(conv)
        except TypeError:
            print("(empty or invalid conversation)")
            return
    for msg in conv:
        if not isinstance(msg, dict):
            continue
        role = msg.get("role", "?")
        content = (msg.get("content") or "").strip()
        label = f"[{role}]"
        if not content:
            print(f"{label} (empty)")
            continue
        # Fill each paragraph to width so long lines wrap
        sub_indent = " " * (len(label) + 1)
        for i, para in enumerate(content.split("\n")):
            if not para.strip():
                print()
                continue
            initial = (label + " ") if i == 0 else sub_indent
            print(textwrap.fill(para, width=width, initial_indent=initial, subsequent_indent=sub_indent))
        print()

In [6]:
# Print conversation of the selected row (edit SELECTED_ROW and re-run)
SELECTED_ROW = 0
if SELECTED_ROW < 0 or SELECTED_ROW >= len(df):
    print(f"SELECTED_ROW {SELECTED_ROW} out of range (0..{len(df)-1})")
else:
    row = df.iloc[SELECTED_ROW]
    cid = row.get("conversation_id", "—")
    print_conversation(row["conversation"], width=100, title=f"--- Row {SELECTED_ROW} (id: {cid}) ---")

--- Row 0 (id: 10517d64ed2898af190f051690a753ca) ---
----------------------------------------------------
[user] Write a product description about a brand Fheys that sell high quality convertible duffle
       bags should be short and concrete no buzz words

[assistant] The Fheys Convertible Duffle Bag is the epitome of high-quality design and
            functionality. Crafted with premium materials, this versatile bag effortlessly adapts to
            your travel needs. Whether you prefer a duffle bag or a backpack, the Fheys Convertible
            Duffle Bag has got you covered.

            The durable construction ensures long-lasting performance, making it the ideal companion
            for your adventures. With ample storage space, you can easily pack all your essentials,
            from clothing to accessories, in an organized manner.

            Featuring sturdy handles and adjustable straps, this bag offers multiple carrying
            options. Whether you want to carry

In [7]:
# Spot-check first N conversations (full conversation wrapped on screen)
SPOT_CHECK_N = 5
for i in range(min(SPOT_CHECK_N, len(df))):
    row = df.iloc[i]
    cid = row.get("conversation_id", "—")
    title = f"--- Conversation {i} (id: {cid}) ---"
    print_conversation(row["conversation"], width=100, title=title)
    print()

--- Conversation 0 (id: 10517d64ed2898af190f051690a753ca) ---
-------------------------------------------------------------
[user] Write a product description about a brand Fheys that sell high quality convertible duffle
       bags should be short and concrete no buzz words

[assistant] The Fheys Convertible Duffle Bag is the epitome of high-quality design and
            functionality. Crafted with premium materials, this versatile bag effortlessly adapts to
            your travel needs. Whether you prefer a duffle bag or a backpack, the Fheys Convertible
            Duffle Bag has got you covered.

            The durable construction ensures long-lasting performance, making it the ideal companion
            for your adventures. With ample storage space, you can easily pack all your essentials,
            from clothing to accessories, in an organized manner.

            Featuring sturdy handles and adjustable straps, this bag offers multiple carrying
            options. Whether

In [8]:
# Spot-check by index (edit SPOT_CHECK_INDICES and re-run to inspect specific rows)
SPOT_CHECK_INDICES = [0, 10, 100]
for i in SPOT_CHECK_INDICES:
    if i < 0 or i >= len(df):
        print(f"--- Index {i} out of range (0..{len(df)-1}) ---")
        continue
    row = df.iloc[i]
    cid = row.get("conversation_id", "—")
    title = f"--- Index {i} (id: {cid}) ---"
    print_conversation(row["conversation"], width=100, title=title)
    print()

--- Index 0 (id: 10517d64ed2898af190f051690a753ca) ---
------------------------------------------------------
[user] Write a product description about a brand Fheys that sell high quality convertible duffle
       bags should be short and concrete no buzz words

[assistant] The Fheys Convertible Duffle Bag is the epitome of high-quality design and
            functionality. Crafted with premium materials, this versatile bag effortlessly adapts to
            your travel needs. Whether you prefer a duffle bag or a backpack, the Fheys Convertible
            Duffle Bag has got you covered.

            The durable construction ensures long-lasting performance, making it the ideal companion
            for your adventures. With ample storage space, you can easily pack all your essentials,
            from clothing to accessories, in an organized manner.

            Featuring sturdy handles and adjustable straps, this bag offers multiple carrying
            options. Whether you want to c

## 0. Topic modeling (NMF)

Landscape of topics from **first user message** (TF-IDF + NMF). The text comes from the **conversation** column: `extract_text_column(df, mode="first_user")` reads each row's `conversation` (list of messages) and takes the first message with `role == "user"` to build `first_user_text`; NMF runs on that. When loading from `data/english_chunks/`, `load_english_chunked_parquet` parses the conversation column from string to list-of-dicts so extraction is not empty. Optionally load saved topic assignments.

In [9]:
# Topic modeling on first user message (no API)
topic_path = Path(INSIGHTS_OUTPUT_DIR) / "topic_assignments.parquet"
texts = df["first_user_text"].fillna("").astype(str)
if LOAD_FROM_SAVED and topic_path.exists():
    topic_df = pd.read_parquet(topic_path)
    df = df.merge(topic_df[["conversation_id", "topic_id"]], on="conversation_id", how="left")
    print("Loaded topic assignments from", topic_path)
else:
    nmf, doc_topics, vectorizer = run_nmf(texts, n_topics=N_TOPICS, random_state=RANDOM_SEED)
    df["topic_id"] = assign_dominant_topic(doc_topics)
    ensure_output_dir(Path(INSIGHTS_OUTPUT_DIR))
    pd.DataFrame({"conversation_id": df["conversation_id"], "topic_id": df["topic_id"]}).to_parquet(topic_path, index=False)
    print("Topic distribution:")
    display(build_topic_distribution_df(doc_topics))
    top_terms = get_top_terms_nmf(nmf, vectorizer, n=10)
    print("Top terms per topic (NMF):")
    for tid, terms in list(top_terms.items())[:10]:
        print(f"  Topic {tid}: {', '.join(terms)}")

Topic distribution:


,topic_id,count,pct
0,0,6424,12.848
1,1,2176,4.352
2,2,940,1.880
3,3,1789,3.578
4,4,9186,18.372
5,5,1415,2.830
6,6,1026,2.052
7,7,473,0.946
8,8,4839,9.678
9,9,1132,2.264


Top terms per topic (NMF):
  Topic 0: ar, prompt, 1, v, description, detailed, 5, prompts, s, imagine
  Topic 1: natsuki, sayori, yuri, monika, s, t, just, mc, clubroom, ll
  Topic 2: hi, gpt, m, chat, help, email, product, tell, thanks, friend
  Topic 3: message, hey, raw, concise, discussion, send, asked, say, short, response
  Topic 4: use, data, text, list, file, want, code, make, language, gpt
  Topic 5: naruto, planet, freedom, lilac, characters, react, sonic, x, girls, carol
  Topic 6: jane, year, old, 14, sam, aaron, animated, teen, ling, dialogue
  Topic 7: hello, world, chatgpt, david, help, cafe, know, say, code, version
  Topic 8: 0, 1, x, const, 2, y, int, 3, n, function
  Topic 9: script, vs, state, conference, make, team, university, video, football, python


In [10]:
# Topic distribution (counts and share)
if "topic_id" in df.columns:
    dist = df["topic_id"].value_counts().sort_index().reset_index()
    dist.columns = ["topic_id", "count"]
    dist["pct"] = 100.0 * dist["count"] / len(df)
    display(dist)

,topic_id,count,pct
0,0.0,1123,2.246
1,1.0,397,0.794
2,2.0,174,0.348
3,3.0,328,0.656
4,4.0,1674,3.348
5,5.0,258,0.516
6,6.0,176,0.352
7,7.0,66,0.132
8,8.0,801,1.602
9,9.0,173,0.346


## 1. Topic / industry and commercial content

**Commercial intent (heuristic only).** The heuristic flags the first user message if it contains any of a fixed set of keywords (e.g. "best", "recommend", "buy", "compare", "review", "top 10", "worth it")—no API call. This gives a quick, noisy signal for product-seeking or commercial intent.

In [ ]:
# Commercial intent: keyword heuristic only (no API)
df = add_commercial_intent_heuristic(df, text_col="first_user_text")
print("Commercial intent (heuristic):", df["has_commercial_intent"].value_counts())

In [ ]:
# Heuristic commercial intent distribution
display(df["has_commercial_intent"].value_counts().to_frame("count"))

## 2. Intent / semantic analysis (themes)

Theme labeling is not run in this notebook (no OpenAI). For faster theme labeling elsewhere, use `label_theme_openai_parallel` in insights_utils (parallel API calls).

In [ ]:
# Theme labeling skipped (no OpenAI in this section). Use label_theme_openai_parallel in insights_utils for faster runs elsewhere.
print("Intent/semantic (theme) labeling not run. For parallel theme labeling see insights_utils.label_theme_openai_parallel.")

In [ ]:
# Theme volume over time and underserved metrics (only if theme column exists, e.g. from label_theme_openai_parallel)
if "theme" in df.columns:
    theme_over_time = theme_counts_over_time(df, label_col="theme", timestamp_col="timestamp", freq="W")
    print("Sample: theme counts by week (first 20 rows):")
    display(theme_over_time.head(20))
    underserved = underserved_metrics(df, label_col="theme", turn_col="turn")
    print("Underserved (conv_share_pct vs turn_share_pct):")
    display(underserved.sort_values("conv_share_pct"))
else:
    print("No theme column; run label_theme_openai_parallel elsewhere to add themes, then re-run this cell.")